# Scaling: Weak Scale

In [ ]:
!lscpu

In [ ]:
!lscpu --all --extended

In [ ]:
from ray import tune
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [ ]:
path = "./ray_results/weak_scaling_/"
analysis = tune.ExperimentAnalysis(experiment_checkpoint_path=path)

In [ ]:
df = analysis.dataframe()
print(f"{len(df)=}")

In [ ]:
df.keys()

In [ ]:
keys = ["ApplyOverSystems/total", "ApplyOverSystemsSeq/total", "UpdateKinematics/total", "ComputeInternalDynamics/total", "UpdateDynamics/total"]
df["Walltime"] = df[keys].sum(1)
df["config/num_rods"] = df["config/z_num_rods_per_thread"] * df["config/num_threads"]

In [ ]:
desire_field = ["Walltime", "config/block_size", "config/num_rods", "config/num_threads", "n_blocks", "config/z_num_rods_per_thread"]
df = df.filter(items=desire_field).dropna()

## Define Functions

In [ ]:
from scipy.optimize import minimize_scalar, curve_fit

def gustafson(x, s):
    return x - s * (x - 1)

def get_serial_fraction(speed_up, func, max_core=64):
    """
    Return S: serial speedup
    """
    speed_up = speed_up[speed_up.index <= max_core]
    
    popt, pcov = curve_fit(func, speed_up.index, speed_up)
    #res = minimize_scalar(func)
    return popt[0]

## Weak Scaling

In [ ]:
group_criteria = ["config/z_num_rods_per_thread"]
d = dict(tuple(df.groupby(group_criteria, dropna=True)))

In [ ]:
problem_sizes = d.keys() # scale: number of rod per thread
problem_sizes

In [ ]:
block_sizes = df["config/block_size"].unique()
block_sizes.sort()
block_sizes

In [ ]:
x_field = "config/num_threads"
y_field = "Walltime"

max_core = 16
elem_per_rod = 16
interpol_x = np.linspace(1, max_core, 100)

for block_size in block_sizes:
    fig = plt.figure(figsize=(12,6))
    ax1 = fig.add_subplot(121)
    ax2 = fig.add_subplot(122)

    best_serial_fraction = 1.0
    for key in d.keys():
        t = d[key].loc[d[key]["config/block_size"]==block_size].groupby(x_field)[y_field]
        t = t.min()
        t = t[t.index <= max_core]
        speed_up = t[2] * t.index / t
        speed_up = speed_up.sort_index()
        efficiency = speed_up / speed_up.index
        ax1.plot(speed_up, label=f"n_elem / thread: 2^{np.log2(key*elem_per_rod)}")
        ax2.plot(100*efficiency, label=f"n_elem / thread: 2^{np.log2(key*elem_per_rod)}")

        serial_fraction = get_serial_fraction(speed_up, func=gustafson, max_core=12)
        best_serial_fraction = min(serial_fraction, best_serial_fraction)

    # Configure Speed Up Plot
    speed_up_pred = gustafson(interpol_x, best_serial_fraction)
    ax1.plot(interpol_x, speed_up_pred, '-.', label=f"best fit S={best_serial_fraction:.02f}")

    ax1.legend()
    ax1.set_xlabel('N Threads')
    ax1.set_ylabel("Run Time / T1")
    ax1.set_title(f"Weak Scale ({block_size=})")
    #ax1.set_xlim(-1,16)

    # Configure Efficiency Plot
    efficiency_pred = speed_up_pred / interpol_x
    ax2.plot(interpol_x, 100*efficiency_pred, '-.', label=f"best fit S={best_serial_fraction:.02f}")
    ax2.legend()
    ax2.set_xlabel("N Threads")
    ax2.set_ylabel("Efficiency %")
    ax2.set_title(f"Efficiency : Weak Scale ({block_size=})")

    print(f"{block_size=}: {best_serial_fraction=}")